#Reto 10 (Prueba 3) - Efecto Magnus

##Jhoan Alejandro Londoño Restrepo

Para un cilindro infinito y rotante sumergido en un fluído incompresible, dibujar el campo de velocidades, las streamlines, los contornos de la función de flujo y la presión alrededor del cilindro.

Con base en el resultado anterior, elabore un código interactivo que permitiendo controlar la velocidad del flujo U, la velocidad de rotación del cilindro y el radio del cilindro, muestre, en la misma gráfica, el campo de velocidades alrededor del cilindro, los contornos de la función de flujo, el campo de presiones y con una flecha la magnitud de la fuerza inducida por efecto Magnus sobre el cilindro.

Para todo ello use los resultados matemáticos de la presentación realizada por Jessica Velásquez.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from matplotlib.patches import Circle

Definimos el campo de velocidades:

In [ ]:
def vfield(x,y,z,t,vpars):
  #Parámetros
  U,a,omega=vpars

  #Calcula coordenadas cilíndricas
  r=np.sqrt(x**2+y**2)
  phi=np.arctan2(y,x)
  #print(phi)

  #Componentes cilíndricas de la velocidad
  vr=U*np.cos(phi)*(1-a**2/r**2)
  vphi = -U*np.sin(phi)*(1+a**2/r**2) + a**2*omega/r
  #print(vr,vphi)

  #Vectores unitarios en coordenadas cilíndricas en cartesianas
  er=np.array([np.cos(phi),np.sin(phi),0])
  ephi=np.array([-np.sin(phi),np.cos(phi),0])
  #print(er,ephi)

  #Velocidad en cartesianas
  v=vr*er+vphi*ephi
  
  return v

In [ ]:
U=10
a=0.5
omega=20

## Campo vectorial instantáneo

In [ ]:
from ipywidgets import interact,fixed,widgets
opciones=dict(continuous_update=False)

def campov(U=10,a=0.5,omega=15):

  vpars=(U,a,omega)
  vfield(20,0,0,1,vpars)

  #Malla rectangular
  NG=30
  xs=np.linspace(-2.5*a,2.5*a,NG)
  ys=np.linspace(-2.5*a,2.5*a,NG)
  zs=np.zeros(NG)

  XS,YS=np.meshgrid(xs,ys)
  ZS=np.zeros_like(XS)

  t=0
  VXS=np.zeros_like(XS)
  VYS=np.zeros_like(XS)
  VZS=np.zeros_like(XS)
  for i in range(NG):
    for j in range(NG):
      if np.sqrt(XS[i,j]**2+YS[i,j]**2)>=a:
        VXS[i,j],VYS[i,j],VZS[i,j]=vfield(XS[i,j],YS[i,j],ZS[i,j],t,vpars)
  plt.figure(figsize=(10,10))
  plt.quiver(XS,YS,VXS,VYS,scale_units="xy",scale=50)
  plt.gca().add_patch(Circle((0,0),a,fill='blue'))
  plt.axis("off")
  
  return

# Código de interacción
interact(campov,
         U=widgets.FloatSlider(min=10,max=50,value=10,step=5,**opciones),
         a=widgets.FloatSlider(min=0.1,max=1.0,value=0.5,step=0.1,**opciones),
         omega=widgets.FloatSlider(min=10,max=50,value=20,step=5,**opciones)
         );

interactive(children=(FloatSlider(value=10.0, continuous_update=False, description='U', max=50.0, min=10.0, st…

# Streamlines

Definimos la ecuación diferencial:

In [ ]:
def streamline(Y,t,t0,vfield,vpars):
  x,y,z=Y

  vx,vy,vz=vfield(x,y,z,t0,vpars)
  
  dxdt=vx
  dydt=vy
  dzdt=vz

  return [dxdt,dydt,dzdt]

La resolvemos para un valor definido de t0 con posiciones iniciales sobre el lado izquierdo e inferior de la malla:

In [ ]:
def campov(U=10,a=0.5,omega=15):

  t0=0
  vpars=(U,a,omega)

  ts = np.linspace(0,10,10000)
  plt.figure(figsize=(10,10))
  x = XS[0,0]
  for y in YS[:,0]:
    Y0=[x,y,0]
    sol=odeint(streamline,Y0,ts,args=(t0,vfield,vpars))
    plt.plot(sol[:,0],sol[:,1],"r-")

  plt.gca().add_patch(Circle((0,0),a,fill='blue'))
  plt.xlim(-2.5*a,2.5*a)
  plt.ylim(-2.5*a,2.5*a)
  plt.axis("off")

  return

# Código de interacción
interact(campov,
         U=widgets.FloatSlider(min=10,max=50,value=10,step=5,**opciones),
         a=widgets.FloatSlider(min=0.1,max=1.0,value=0.5,step=0.1,**opciones),
         omega=widgets.FloatSlider(min=10,max=50,value=20,step=5,**opciones)
         );

interactive(children=(FloatSlider(value=10.0, continuous_update=False, description='U', max=50.0, min=10.0, st…

## Contornos de la función de corriente

Vimos en la clase para el cilindro no rotante que la ecuación de la función de corriente para un cilindro no rotante es:

$$ \psi = - \int v_{\phi} dr + C_\phi$$

Ahora agreguemos el término cuando el cilindro rota: 

$$ \psi = \int r v_{r} d \phi + C_r$$

Esta integrales son fáciles de calcular mediante el uso de integral simbólica de sympy

In [ ]:
import sympy as sp
from sympy import pi
from sympy import oo, exp, Symbol, integrate

phi = Symbol('phi')
r = Symbol('r')

velr = U*sp.cos(phi)*(1-a**2/r**2)
velphi = -U*sp.sin(phi)*(1+a**2/r**2) + a**2*omega/r 

integrate(r*velr,phi)

10*r*(1 - 0.25/r**2)*sin(phi)

In [ ]:
-integrate(velphi,r)

10.0*r*sin(phi) - 5.0*log(r) - 2.5*sin(phi)/r

Por tanto:

$$ \psi = Ur \sin{\phi} - \frac{U\sin{\phi}}{r} - U\log{r}$$.


In [ ]:
def campov(U=10,a=0.5,omega=15):

  #Malla rectangular
  NG=100
  xs=np.linspace(-5*a,5*a,NG)
  ys=np.linspace(-5*a,5*a,NG)
  zs=np.zeros(NG)

  XS,YS=np.meshgrid(xs,ys)
  ZS=np.zeros_like(XS)

  #Función de corriente
  RS=np.sqrt(XS**2+YS**2)
  PHIS=np.arctan2(YS,XS)

  PSIS=U*RS*np.sin(PHIS)-U*np.log(RS)-(U*np.sin(PHIS)/RS)

  plt.figure(figsize=(10,10))
  plt.contour(XS,YS,PSIS,levels=100)
  plt.gca().add_patch(Circle((0,0),a,fill='blue',zorder=100))
  plt.axis("off")
  plt.show()

  return

# Código de interacción
interact(campov,
         U=widgets.FloatSlider(min=10,max=50,value=10,step=5,**opciones),
         a=widgets.FloatSlider(min=0.1,max=1.0,value=0.5,step=0.1,**opciones),
         omega=widgets.FloatSlider(min=10,max=50,value=20,step=5,**opciones)
         );

interactive(children=(FloatSlider(value=10.0, continuous_update=False, description='U', max=50.0, min=10.0, st…

## Contornos de la presión

In [ ]:
def campov(U=10,a=0.5,omega=15):
  #Malla rectangular
  NG=30
  xs=np.linspace(-2.5*a,2.5*a,NG)
  ys=np.linspace(-2.5*a,2.5*a,NG)
  zs=np.zeros(NG)

  XS,YS=np.meshgrid(xs,ys)
  ZS=np.zeros_like(XS)

  #Función de corriente
  RS=np.sqrt(XS**2+YS**2)
  PHIS=np.arctan2(YS,XS)


  rho0=1
  C = a**2*omega
  PS_0=0.5*rho0*U**2*a**2/RS**2*(4*np.cos(PHIS)**2-2-a**2/RS**2) #presión del fluido cuando el cilindro esta quieto
  PS_rot = 0.5*rho0*C*U*np.sin(PHIS)*(1+a**2/RS**2)/RS - 0.5*C**2/RS**2 #presión del fluido cuando el cilindro esta rotando
  PS = PS_0+PS_rot 
  Pd=rho0*U**2/2

  plt.figure(figsize=(10,10))
  #plt.contourf(XS,YS,PS,levels=np.linspace(PS.min(),PS.max(),100))
  #plt.contour(XS,YS,PS,levels=1000)
  #plt.contourf(XS,YS,np.log10(np.abs(PS)),levels=100)
  c=plt.contourf(XS,YS,PS,levels=np.linspace(-6*Pd,Pd,100),cmap="Spectral")
  plt.colorbar(c)
  plt.gca().add_patch(Circle((0,0),a,fill='blue',zorder=100))
  plt.axis("equal")
  plt.axis("off")

  return

# Código de interacción
interact(campov,
         U=widgets.FloatSlider(min=10,max=50,value=10,step=5,**opciones),
         a=widgets.FloatSlider(min=0.1,max=1.0,value=0.5,step=0.1,**opciones),
         omega=widgets.FloatSlider(min=10,max=50,value=20,step=5,**opciones)
         );

interactive(children=(FloatSlider(value=10.0, continuous_update=False, description='U', max=50.0, min=10.0, st…

##Todas las gráficas juntas

In [ ]:
def campov(U=10,a=0.5,omega=15):

  vpars=(U,a,omega)
  vfield(20,0,0,1,vpars)

  #Malla rectangular
  NG=30
  xs=np.linspace(-5*a,5*a,NG)
  ys=np.linspace(-5*a,5*a,NG)
  zs=np.zeros(NG)

  XS,YS=np.meshgrid(xs,ys)
  ZS=np.zeros_like(XS)

  t=0
  VXS=np.zeros_like(XS)
  VYS=np.zeros_like(XS)
  VZS=np.zeros_like(XS)
  for i in range(NG):
    for j in range(NG):
      if np.sqrt(XS[i,j]**2+YS[i,j]**2)>=a:
        VXS[i,j],VYS[i,j],VZS[i,j]=vfield(XS[i,j],YS[i,j],ZS[i,j],t,vpars)
  plt.figure(figsize=(10,10))
  plt.quiver(XS,YS,VXS,VYS,scale_units="xy",scale=60)
  plt.gca().add_patch(Circle((0,0),a,fill='blue'))
  plt.axis("off")

  t0=0
  vpars=(U,a,omega)

  ts = np.linspace(0,10,10000)
  x = XS[0,0]
  for y in YS[:,0]:
    Y0=[x,y,0]
    sol=odeint(streamline,Y0,ts,args=(t0,vfield,vpars))
    plt.plot(sol[:,0],sol[:,1],"r-")

  plt.gca().add_patch(Circle((0,0),a,fill='blue'))

  #Malla rectangular
  NG=100
  xs=np.linspace(-5*a,5*a,NG)
  ys=np.linspace(-5*a,5*a,NG)
  zs=np.zeros(NG)

  XS,YS=np.meshgrid(xs,ys)
  ZS=np.zeros_like(XS)

  #Función de corriente
  RS=np.sqrt(XS**2+YS**2)
  PHIS=np.arctan2(YS,XS)

  PSIS=U*RS*np.sin(PHIS)-U*np.log(RS)-(U*np.sin(PHIS)/RS)

  plt.contour(XS,YS,PSIS,levels=150)
  plt.gca().add_patch(Circle((0,0),a,fill='blue',zorder=100))
  plt.axis("off")

  #Malla rectangular
  NG=30
  xs=np.linspace(-5*a,5*a,NG)
  ys=np.linspace(-5*a,5*a,NG)
  zs=np.zeros(NG)

  XS,YS=np.meshgrid(xs,ys)
  ZS=np.zeros_like(XS)

  #Función de corriente
  RS=np.sqrt(XS**2+YS**2)
  PHIS=np.arctan2(YS,XS)


  rho0=1
  C = a**2*omega
  PS_0=0.5*rho0*U**2*a**2/RS**2*(4*np.cos(PHIS)**2-2-a**2/RS**2)
  PS_in = 0.5*rho0*C*U*np.sin(PHIS)*(1+a**2/RS**2)/RS - 0.5*C**2/RS**2
  PS = PS_0+PS_in
  Pd=rho0*U**2/2

  #plt.contourf(XS,YS,PS,levels=np.linspace(PS.min(),PS.max(),100))
  #plt.contour(XS,YS,PS,levels=1000)
  #plt.contourf(XS,YS,np.log10(np.abs(PS)),levels=100)
  c=plt.contourf(XS,YS,PS,levels=np.linspace(-6*Pd,Pd,100),cmap="Spectral")
  plt.colorbar(c)
  plt.gca().add_patch(Circle((0,0),a,fill='blue',zorder=100))
  plt.axis("equal")
  plt.axis("off")
  plt.quiver(XS,YS,VXS,VYS,scale_units="xy",scale=50)
  plt.xlim(-2.5*a,2.5*a)
  plt.ylim(-2.5*a,2.5*a)
  
  return

# Código de interacción
interact(campov,
         U=widgets.FloatSlider(min=10,max=50,value=10,step=5,**opciones),
         a=widgets.FloatSlider(min=0.1,max=1.0,value=0.5,step=0.1,**opciones),
         omega=widgets.FloatSlider(min=10,max=50,value=20,step=5,**opciones)
         );

interactive(children=(FloatSlider(value=10.0, continuous_update=False, description='U', max=50.0, min=10.0, st…